# CAT-5 — Cross-catalog federation: Nessie staging → Glue marts

Modern lakehouses fan out into multiple catalogs on purpose: **staging** on a catalog that supports branching (so ETL engineers can rehearse changes on `dev` before touching `main`), **marts** on a governed catalog with a real metastore (so downstream BI, dbt, and Trino can trust the schema and coordinate concurrent writers).

This notebook runs that split as a single Spark SQL job:

```
 nessie_catalog.staging.orders ──► glue_catalog.marts.orders_daily
 (Nessie REST + branching — CAT-3) (AWS Glue Data Catalog — real metastore)
```

**Break → Do → Detect → Prove:**
1. **Do** — CREATE both namespaces (one per catalog). Seed 5 orders into Nessie staging. Aggregate into Glue marts via a **CTAS** that reads from one catalog and writes to another — one job, two catalogs.
2. **Detect** — read back from Glue; the daily totals should match by-hand math.
3. **Prove** — the marts data physically landed under `s3://warehouse/glue/…` while the staging data sits under `s3://warehouse/nessie/…`. Two separate S3 trees, one query planner.

> **Why Glue and not Polaris?** MiniStack Community's STS emulator can't vend usable subscoped S3 credentials, and Polaris's data-plane write path requires that AssumeRole loop. Glue works because the Iceberg AWS bundle uses **static credentials** and skips STS entirely — see `docs/PLAN_V2.md` §WS1 for the full RCA. Polaris stays governance-only in this repo ([CAT-2](cat2_polaris_rbac.ipynb)).

**Prereqs:** `make up && make catalogs-up`. Both `glue_catalog` and `nessie_catalog` are pre-configured in `conf/spark-defaults.conf`.

In [1]:
from common.spark_session import spark
# Public helpers from common.table_meta — thin S3 boilerplate the whole repo shares.
# (No underscore-prefixed imports: the *technique* — federated CTAS — stays inline.)
from common.table_meta import s3_client, split_s3, wipe_prefix

STAGING = "nessie_catalog.staging.orders"        # branch-friendly source
MARTS   = "glue_catalog.marts.orders_daily"      # governed destination

# Idempotent reset. Nessie is atomic — DROP TABLE IF EXISTS is enough.
# Glue holds a metastore pointer; Iceberg's DROP TABLE removes the pointer,
# but not the S3 files (an intentional Iceberg design — no accidental data
# loss). We wipe the S3 prefix ourselves so the CREATE TABLE below starts clean.
try:
    spark.sql("USE REFERENCE main IN nessie_catalog").collect()
except Exception:
    pass
for t in (STAGING, MARTS):
    try:
        spark.sql(f"DROP TABLE IF EXISTS {t}").collect()
    except Exception as e:
        # Stale-pointer tolerance (see CAT-3): a prior run's metadata.json may
        # have been wiped from S3 while its pointer lingers in the catalog.
        print(f"tolerated: {type(e).__name__} on DROP {t}: {str(e)[:80]}")
removed = wipe_prefix("s3a://warehouse/glue/marts.db/orders_daily/")
print(f"reset done — cleared {removed} object(s) under s3://warehouse/glue/marts.db/orders_daily/")

reset done — cleared 0 object(s) under s3://warehouse/glue/marts.db/orders_daily/


## 1. Two namespaces, one per catalog

Namespaces are catalog-local — `nessie_catalog.staging` and `glue_catalog.marts` are unrelated to each other, even if named similarly. Each catalog owns its own namespace tree, and each `CREATE NAMESPACE` goes through the catalog's own creation code path (Nessie commits a ref-store row; Glue writes a database entry to the Glue metastore).

In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie_catalog.staging").collect()
spark.sql("CREATE NAMESPACE IF NOT EXISTS glue_catalog.marts").collect()

print("nessie_catalog namespaces:",
      [r[0] for r in spark.sql("SHOW NAMESPACES IN nessie_catalog").collect()])
print("glue_catalog namespaces  :",
      [r[0] for r in spark.sql("SHOW NAMESPACES IN glue_catalog").collect()])

nessie_catalog namespaces: ['marts', 'staging']
glue_catalog namespaces  : ['marts']


## 2. Seed the branch-friendly source on Nessie

Five orders across two days — small enough to hand-verify the aggregate. Nessie catalogs like this one are where a real ETL team stages incoming data: engineers can `CREATE BRANCH dev`, run migrations/backfills/schema evolution, and merge (that's the entire [CAT-3](nessie/cat3_branching.ipynb) lesson). Marts, meanwhile, live on the *governed* side and stay immutable to that experimentation.

In [3]:
spark.sql(f"""
CREATE TABLE {STAGING} (
    order_id BIGINT,
    ts       TIMESTAMP,
    amount   DOUBLE
) USING iceberg
""").collect()

spark.sql(f"""
INSERT INTO {STAGING} VALUES
    (1, timestamp'2026-07-01 10:00:00', 100.0),
    (2, timestamp'2026-07-01 15:00:00',  50.0),
    (3, timestamp'2026-07-02 09:00:00',  75.0),
    (4, timestamp'2026-07-02 14:00:00',  25.0),
    (5, timestamp'2026-07-02 20:00:00', 200.0)
""").collect()

print(f"staging on nessie_catalog: {STAGING}")
spark.sql(f"SELECT * FROM {STAGING} ORDER BY order_id").show(truncate=False)

staging on nessie_catalog: nessie_catalog.staging.orders
+--------+-------------------+------+
|order_id|ts                 |amount|
+--------+-------------------+------+
|1       |2026-07-01 10:00:00|100.0 |
|2       |2026-07-01 15:00:00|50.0  |
|3       |2026-07-02 09:00:00|75.0  |
|4       |2026-07-02 14:00:00|25.0  |
|5       |2026-07-02 20:00:00|200.0 |
+--------+-------------------+------+



## 3. The federated CTAS — Nessie → Glue in one query

This is the payoff: **one Spark SQL statement, two catalog contact points**. Spark's Catalyst planner resolves each fully-qualified table name through its own catalog implementation — Nessie's `NessieCatalog` REST client for the source, Iceberg's `GlueCatalog` for the destination — inside a single job. No ETL glue code, no intermediate staging table, no `df.write.saveAsTable(...)` shuffle back to the driver.

`CREATE TABLE ... AS SELECT` lets the destination catalog pick the target schema from the aggregate's output types, avoiding a manual `CREATE` + `INSERT` column-order skew.

In [4]:
spark.sql(f"""
CREATE TABLE {MARTS}
USING iceberg
AS
SELECT date_trunc('day', ts) AS day,
       SUM(amount)           AS total,
       COUNT(*)              AS order_count
FROM {STAGING}
GROUP BY 1
ORDER BY 1
""").collect()

print(f"materialised marts on glue_catalog: {MARTS}")

materialised marts on glue_catalog: glue_catalog.marts.orders_daily


## 4. Detect — read the marts back and check the arithmetic

The two days should give us `2026-07-01: total 150.0 (100+50), count 2` and `2026-07-02: total 300.0 (75+25+200), count 3`. Assert hard — if the federation lost any rows or scrambled the aggregation, this cell fails loudly.

In [5]:
mart_rows = spark.sql(f"SELECT * FROM {MARTS} ORDER BY day").collect()
print(f"marts rows ({len(mart_rows)}):")
for r in mart_rows:
    print("  ", r.asDict())

expected = {"2026-07-01": (150.0, 2), "2026-07-02": (300.0, 3)}
by_day = {r["day"].strftime("%Y-%m-%d"): (float(r["total"]), r["order_count"])
          for r in mart_rows}

print()
for day, (want_total, want_count) in expected.items():
    got_total, got_count = by_day[day]
    assert got_total == want_total and got_count == want_count, (
        f"{day}: got total={got_total} count={got_count}, want total={want_total} count={want_count}"
    )
    print(f"  OK  {day}: total={got_total} (want {want_total}), count={got_count} (want {want_count})")
print("\nAll aggregates match by-hand math — federation preserved the arithmetic across two catalog boundaries.")

marts rows (2):
   {'day': datetime.datetime(2026, 7, 1, 5, 30), 'total': 150.0, 'order_count': 2}
   {'day': datetime.datetime(2026, 7, 2, 5, 30), 'total': 300.0, 'order_count': 3}

  OK  2026-07-01: total=150.0 (want 150.0), count=2 (want 2)
  OK  2026-07-02: total=300.0 (want 300.0), count=3 (want 3)

All aggregates match by-hand math — federation preserved the arithmetic across two catalog boundaries.


## 5. Prove — the physical S3 layout

Nothing in the SQL mentioned S3 paths — each catalog resolved its own physical layout. Listing S3 shows the two totally separate object trees:

- `s3://warehouse/nessie/staging.db/orders/…` — Nessie's warehouse, path controlled by `nessie_catalog.warehouse`
- `s3://warehouse/glue/marts.db/orders_daily/…` — Glue's warehouse, path controlled by `glue_catalog.warehouse`

That physical separation is what makes it safe to give two different teams different write/RBAC boundaries: they don't step on each other's keys.

In [6]:
s3 = s3_client()
bucket, _ = split_s3("s3a://warehouse/")

def list_under(prefix, limit=8):
    print(f"\ns3://{bucket}/{prefix}")
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    keys = [o["Key"] for o in resp.get("Contents", [])]
    if not keys:
        print("   (empty)")
        return
    for k in sorted(keys)[:limit]:
        # A short tag for the file kind
        if k.endswith(".parquet"):    tag = "data "
        elif k.endswith(".avro"):     tag = "mnfst"
        elif k.endswith(".json"):     tag = "meta "
        else:                          tag = "     "
        print(f"   [{tag}] {k}")
    if len(keys) > limit:
        print(f"   …and {len(keys) - limit} more")

print("Two physical prefixes, one federated query:")
list_under("nessie/")
list_under("glue/marts.db/orders_daily/")

# Hard assertion: both trees have Parquet data + Iceberg metadata
def has_files(prefix, suffix):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    return any(o["Key"].endswith(suffix) for o in resp.get("Contents", []))

assert has_files("nessie/", ".parquet"), "Nessie staging has no data files on S3"
assert has_files("glue/marts.db/orders_daily/", ".parquet"), "Glue marts has no data files on S3"
assert has_files("glue/marts.db/orders_daily/", ".metadata.json"), "Glue marts has no Iceberg metadata"
print("\nAssertions OK — both catalogs' physical layouts are on S3 as expected.")

Two physical prefixes, one federated query:

s3://warehouse/nessie/
   [data ] nessie/marts/cat1_orders_855b3c1d-6dad-43df-9fb9-e68a572710e6/data/00000-3-c789630d-1c20-460e-b05a-e8e803e4ef8e-0-00001.parquet
   [data ] nessie/marts/cat1_orders_855b3c1d-6dad-43df-9fb9-e68a572710e6/data/00001-4-c789630d-1c20-460e-b05a-e8e803e4ef8e-0-00001.parquet
   [meta ] nessie/marts/cat1_orders_855b3c1d-6dad-43df-9fb9-e68a572710e6/metadata/00000-a8e71fb0-e590-4d67-a6a7-4ace3c681c7d.metadata.json
   [meta ] nessie/marts/cat1_orders_855b3c1d-6dad-43df-9fb9-e68a572710e6/metadata/00001-f3d2e21e-3b73-4a6f-b770-4c1b8d2459f3.metadata.json
   [mnfst] nessie/marts/cat1_orders_855b3c1d-6dad-43df-9fb9-e68a572710e6/metadata/39a4cb96-c05f-4c90-816e-c030521e92a6-m0.avro
   [mnfst] nessie/marts/cat1_orders_855b3c1d-6dad-43df-9fb9-e68a572710e6/metadata/snap-393379293653393052-1-39a4cb96-c05f-4c90-816e-c030521e92a6.avro
   [data ] nessie/staging/orders_496c3241-ab84-48b2-9d8f-8938adc41683/data/00000-7-e2facb44-9c90-4d

## What you just saw

- **A single Spark job can address every catalog registered in `spark-defaults.conf`.** The catalog type (REST for Nessie, Glue metastore for Iceberg-on-Glue) is transparent to the SQL — the planner just calls each catalog's implementation.
- **A federated CTAS is one query.** `CREATE TABLE glue_catalog.marts.x AS SELECT … FROM nessie_catalog.staging.y` reads from one catalog and writes to another in one atomic step — no shuffling through the driver, no intermediate table.
- **The physical layout mirrors the logical split.** Nessie staging under `s3://warehouse/nessie/`, Glue marts under `s3://warehouse/glue/`. Different teams / RBAC roles / retention policies can target the different prefixes cleanly.
- **Governance destination = Glue in this repo.** Polaris stays governance-only ([CAT-2](cat2_polaris_rbac.ipynb)) because MiniStack Community can't vend STS credentials for its write path. In a real cloud environment (or LocalStack/MiniStack Pro), swap `glue_catalog` → `polaris_catalog` and everything above is identical.

### Try it yourself

- Branch the staging source: `CREATE BRANCH dev IN nessie_catalog FROM main`, insert dev-only rows, `MERGE`, then re-run the CTAS — you'll see the aggregate update. That's the whole ETL rehearsal loop from [CAT-3](nessie/cat3_branching.ipynb).
- Repoint the destination: swap `glue_catalog.marts.orders_daily` for `iceberg_catalog.marts.orders_daily` (the Hadoop anti-pattern from LAK-10) and observe that the SQL is unchanged — only the physical layout under `s3://warehouse/iceberg/marts/` differs.
- Add a Trino service in front of `glue_catalog` and query `marts.orders_daily` from Trino: same table, different engine — that's what "governed" buys you.

**Related:** [CAT-1](nessie/cat1_nessie_intro.ipynb) (pointer ownership) · [CAT-2](cat2_polaris_rbac.ipynb) (RBAC on Polaris) · [CAT-3](nessie/cat3_branching.ipynb) (branch-merge on Nessie) · [`catalog_api_playground.ipynb`](catalog_api_playground.ipynb) (raw REST + OAuth plumbing).

## FAQ — Why does the Glue explorer show an empty `marts` database after running?

That's the *correct* behavior. `DROP TABLE` removes the Iceberg pointer from the Glue metastore (so `GetTables(marts)` returns `[]`), but leaves the Parquet + `metadata.json` files on S3 (Iceberg refuses to purge data on drop unless you use `DROP TABLE … PURGE`). If you want to browse a live table after the lesson, comment out the final `DROP TABLE` cell below and re-run — the aggregate stays visible in StackPort (`s3://warehouse/glue/marts.db/orders_daily/`) and in Nimtable/Superset until you drop it or `make clean`.

## Teardown

Drop both tables. `make clean` clears MiniStack for a fresh warehouse. Note: Iceberg's `DROP TABLE` deliberately leaves the physical Parquet + metadata on S3 (no accidental data loss); use `DROP TABLE … PURGE` if the engine supports it, or `wipe_prefix()` for a nuclear reset.

In [7]:
for t in (MARTS, STAGING):
    try:
        spark.sql(f"DROP TABLE IF EXISTS {t}").collect()
        print(f"dropped {t}")
    except Exception as e:
        print(f"tolerated: {type(e).__name__} on DROP {t}")

dropped glue_catalog.marts.orders_daily
dropped nessie_catalog.staging.orders
